In [3]:
import pandas as pd
def load_data(interference_file_name: str) -> pd.DataFrame:
    interference_df = pd.read_csv(interference_file_name, low_memory=False)

    # Encode info, channel columns to dummies
    columns_to_encode = ['info', 'channel']

    for col in columns_to_encode:
        interference_df = pd.get_dummies(interference_df, columns=[col], prefix='is')

    # Keep all relevant columns having an is_* prefix, drop the rest
    # use this for Samsung-trained models
    channels_to_keep = ['is_37', 'is_38', 'is_39', 'is_ADV_IND', 'is_ADV_NONCONN_IND', 'is_ADV_SCAN_IND', 'is_SCAN_REQ', 'is_adv_channel' ,'is_broadcast']
    # use this for Apple-trained models
    #channels_to_keep = ['is_37', 'is_38', 'is_39', 'is_ADV_IND', 'is_ADV_NONCONN_IND', 'is_SCAN_REQ', 'is_ADV_EXT_IND', 'is_ADV_SCAN_IND', 'is_AUX_ADV_IND', 'is_adv_channel' ,'is_broadcast']
    other_channel_cols = [col for col in interference_df.columns if col.startswith('is_') and col not in channels_to_keep]
    interference_df = interference_df.drop(columns=other_channel_cols)
    # add missing dummy channels
    missing_channels = [ch for ch in channels_to_keep if ch not in interference_df.columns]
    for ch in missing_channels:
        interference_df[ch] = False



    return interference_df

In [4]:
def combine_data(interference_df: pd.DataFrame, target_labels: list) -> pd.DataFrame:
    # Aggregate dataframe by source and calculate new columns
    interfered_df_agg = aggregate_features('source', interference_df)
    interfered_df_calc = calculate_features(interference_df, interfered_df_agg)

    interfered_df_calc['group_label'] = interfered_df_calc['labelled_device'].apply(
        lambda x: 1 if x in target_labels else 0
    )

    return interfered_df_calc

In [5]:
# Group dataframe by source and extract some metrics
def aggregate_features(group_filter: str, input_df: pd.DataFrame) -> pd.DataFrame:
    input_df = input_df.sort_values(by=['source', 'time'])
    input_df['time_diff'] = input_df.groupby('source')['time'].diff()

    agg_df = input_df.groupby(group_filter).agg(
        ad_type_mean_length=('ad_type', lambda x: x.str.len().mean()),
        data_mean_length=('data', lambda x: x.str.len().mean()),
        uuid16_mean_length=('uuid16', lambda x: x.astype(str).str.len().mean()), # CHANGED: all values are NAN, so x.str gives an error
        service_data_mean_length=('service_data', lambda x: x.astype(str).str.len().mean()),
        mean_time_between_packets=('time_diff', 'mean'),
    )

    return agg_df

In [6]:
# Calculate some source-based feature columns
def calculate_features(actual_df: pd.DataFrame, aggregated_df: pd.DataFrame) -> pd.DataFrame:
    actual_df = actual_df.sort_values(by=['source', 'time'])
    actual_df['time_to_next'] = actual_df.groupby('source')['time'].shift(-1) - actual_df['time']
    actual_df['time_to_next'] = actual_df['time_to_next'].fillna(0)
    actual_df = actual_df.sort_values(by=['source', 'time'])
    actual_df['time_diff'] = actual_df.groupby('source')['time'].diff()

    group_means = aggregated_df['mean_time_between_packets']
    actual_df['mean_time_between_packets'] = actual_df['source'].map(group_means)
    actual_df['mean_time_between_packets'] = actual_df['mean_time_between_packets'].fillna(0)

    sd_group_means = aggregated_df['service_data_mean_length']
    actual_df['service_data_mean_length'] = actual_df['source'].map(sd_group_means)
    actual_df['service_data_mean_length'] = actual_df['service_data_mean_length'].fillna(0)

    actual_df['packet_send_freq'] = actual_df['time_to_next'] / actual_df['mean_time_between_packets']
    actual_df['packet_send_freq'] = actual_df['packet_send_freq'].fillna(0)

    actual_df['service_data_len_ratio'] = actual_df['service_data'].str.len() / actual_df['service_data_mean_length']
    actual_df['service_data_len_ratio'] = actual_df['service_data_len_ratio'].fillna(0)

    ad_type_group_means = aggregated_df['ad_type_mean_length']
    actual_df['ad_type_mean_length'] = actual_df['source'].map(ad_type_group_means)
    actual_df['ad_type_mean_length'] = actual_df['ad_type_mean_length'].fillna(0)
    actual_df['ad_type_len_ratio'] = actual_df['ad_type'].str.len() / actual_df['ad_type_mean_length']
    actual_df['ad_type_len_ratio'] = actual_df['ad_type_len_ratio'].fillna(0)

    data_group_means = aggregated_df['data_mean_length']
    actual_df['data_mean_length'] = actual_df['source'].map(data_group_means)
    actual_df['data_mean_length'] = actual_df['data_mean_length'].fillna(0)
    actual_df['data_len_ratio'] = actual_df['data'].str.len() / actual_df['data_mean_length']
    actual_df['data_len_ratio'] = actual_df['data_len_ratio'].fillna(0)

    uuid16_group_means = aggregated_df['uuid16_mean_length']
    actual_df['uuid16_mean_length'] = actual_df['source'].map(uuid16_group_means)
    actual_df['uuid16_mean_length'] = actual_df['uuid16_mean_length'].fillna(0)
    actual_df['uuid16_len_ratio'] = actual_df['uuid16'].astype(str).str.len() / actual_df['uuid16_mean_length']
    actual_df['uuid16_len_ratio'] = actual_df['uuid16_len_ratio'].fillna(0)

    # Drop intermediate columns as they were only used to help the calculation
    actual_df = actual_df.drop(columns=['mean_time_between_packets', 'time_diff', 'service_data_mean_length', 'ad_type_mean_length', 'data_mean_length', 'uuid16_mean_length'])

    return actual_df

In [6]:
# Ablation Study for Apple model features
# Place this in AblationStudy.ipynb (single cell)

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit

# -----------------------------
# 1) Build final modeling table
# -----------------------------
# Assumes these functions already exist (from modeling_apple.ipynb):
# load_data, combine_data, aggregate_features, calculate_features
target_list_apple = ['iPad', 'AirTag', 'AirPods']

apart_df_1 = load_data('../data/apple_labeled/apple_apartment_group1_6h_1_labeled.csv')
apart_df_2 = load_data('../data/apple_labeled/apple_apartment_group1_6h_2_labeled.csv')
apart_df_3 = load_data('../data/apple_labeled/apple_apartment_group1_6h_3_labeled.csv')
df_eval = load_data('../data/apple_labeled/apple_uni_group1_2h_labeled.csv')

df_eval["service_data"] = df_eval["service_data"].astype(apart_df_1["service_data"].dtype)

int_apart_1 = combine_data(apart_df_1, target_list_apple)
int_apart_2 = combine_data(apart_df_2, target_list_apple)
int_apart_3 = combine_data(apart_df_3, target_list_apple)
int_eval = combine_data(df_eval, target_list_apple)

train_df = pd.concat([int_apart_1, int_apart_2, int_apart_3], ignore_index=True)

# Use your existing dropped columns if you already defined them in modeling_apple
# Otherwise set a safe default:
try:
    dropped_columns
except NameError:
    dropped_columns = [
        'source', 'labelled_device', 'time', 'ad_type', 'data', 'service_data',
        'uuid16', 'uuid128', 'manufacturer_data', 'rssi'
    ]

# Keep only numeric/bool model-ready columns
base_X = train_df.drop(columns=[c for c in dropped_columns if c in train_df.columns] + ['group_label'], errors='ignore')
base_X = base_X.select_dtypes(include=['number', 'bool']).copy()
y = train_df['group_label'].astype(int)
groups = train_df['source'] if 'source' in train_df.columns else pd.Series(np.arange(len(train_df)))

# -----------------------------
# 2) Define ablation groups
# -----------------------------
feature_groups = {
    "channel+packet_type_flags": [
        c for c in base_X.columns if c.startswith('is_')
    ],
    "length_ratio_features": [
        c for c in base_X.columns if 'len_ratio' in c or 'len_' in c
    ],
    "timing_features": [
        c for c in base_X.columns if (
            'time' in c or
            'delta_' in c or
            'packet_send_freq' in c or
            'packets_within_' in c
        )
    ],
}

# Make groups unique / existent
for k, v in feature_groups.items():
    feature_groups[k] = sorted(list(set([c for c in v if c in base_X.columns])))

# -----------------------------
# 3) One split + train/eval fn
# -----------------------------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(base_X, y, groups=groups))

X_train_full = base_X.iloc[train_idx]
X_test_full = base_X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

def train_eval(X_train, X_test, y_train, y_test):
    clf = RandomForestClassifier(
        random_state=1,
        class_weight='balanced',
        n_estimators=50,
        min_samples_split=10,
        min_samples_leaf=3,
        min_weight_fraction_leaf=0.002
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average='binary', zero_division=0)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

# -----------------------------
# 4) Baseline + ablations
# -----------------------------
rows = []

# Baseline
baseline_metrics = train_eval(X_train_full, X_test_full, y_train, y_test)
rows.append({
    "setting": "baseline_all_features",
    "removed_group": "",
    "n_features": X_train_full.shape[1],
    **baseline_metrics
})

# Ablation: remove one group at a time
for group_name, cols_to_remove in feature_groups.items():
    if len(cols_to_remove) == 0:
        continue
    keep_cols = [c for c in base_X.columns if c not in cols_to_remove]
    X_train = X_train_full[keep_cols]
    X_test = X_test_full[keep_cols]

    m = train_eval(X_train, X_test, y_train, y_test)
    rows.append({
        "setting": f"ablate_{group_name}",
        "removed_group": group_name,
        "n_features": X_train.shape[1],
        **m
    })

results = pd.DataFrame(rows).sort_values("f1", ascending=False).reset_index(drop=True)

# Add deltas vs baseline
for metric in ["accuracy", "precision", "recall", "f1"]:
    results[f"delta_{metric}_vs_baseline"] = results[metric] - baseline_metrics[metric]

pd.set_option('display.max_colwidth', 120)
display(results)

# Optional: persist
results.to_csv("../code/apple_ablation_results.csv", index=False)
print("Saved: ../code/apple_ablation_results.csv")

,setting,removed_group,n_features,accuracy,precision,recall,f1,delta_accuracy_vs_baseline,delta_precision_vs_baseline,delta_recall_vs_baseline,delta_f1_vs_baseline
0,baseline_all_features,,26,0.997613,0.997824,0.999075,0.998449,0.000000,0.000000,0.000000,0.000000
1,ablate_channel+packet_type_flags,channel+packet_type_flags,17,0.993040,0.993708,0.997266,0.995484,-0.004573,-0.004116,-0.001809,-0.002965
2,ablate_length_ratio_features,length_ratio_features,18,0.992707,0.998095,0.992413,0.995246,-0.004906,0.000271,-0.006662,-0.003203
3,ablate_timing_features,timing_features,21,0.992035,0.997627,0.992006,0.994809,-0.005577,-0.000197,-0.007069,-0.003641


Saved: ../code/apple_ablation_results.csv


In [7]:
# 1) Inspect all model features actually used
all_features = list(base_X.columns)
print(f"Total features: {len(all_features)}")
for f in all_features:
    print(f)

# 2) Richer feature-group map (covers more families seen in modeling_apple)
feature_groups = {
    "channel_packet_flags": [c for c in all_features if c.startswith("is_")],
    "presence_flags": [c for c in all_features if c.startswith("has_")],
    "time_to_next_and_freq": [c for c in all_features if c in ["time_to_next", "packet_send_freq", "time_to_next_binned", "packet_send_freq_binned"]],
    "burst_density": [c for c in all_features if c.startswith("packets_within_")],
    "delta_time_bins": [c for c in all_features if c.startswith("delta_") and c.endswith("_binned")],
    "time_window_bins": [c for c in all_features if "time_start_end_binned" in c],
    "length_bins": [c for c in all_features if c in ["len_ad_type_binned", "len_data_binned"]],
    "length_ratios_raw": [c for c in all_features if c in ["service_data_len_ratio", "ad_type_len_ratio", "data_len_ratio", "uuid16_len_ratio"]],
    "length_ratios_binned": [c for c in all_features if c.endswith("_len_ratio_binned")],
}

# keep only non-empty groups
feature_groups = {k: sorted(set(v)) for k, v in feature_groups.items() if len(v) > 0}

print("\nFeature groups used for ablation:")
for k, v in feature_groups.items():
    print(f"- {k}: {len(v)} features")

# 3) Run drop-one-group ablation for all groups above
rows = []

baseline_metrics = train_eval(X_train_full, X_test_full, y_train, y_test)
rows.append({
    "setting": "baseline_all_features",
    "removed_group": "",
    "n_features": X_train_full.shape[1],
    **baseline_metrics
})

for group_name, cols_to_remove in feature_groups.items():
    keep_cols = [c for c in all_features if c not in cols_to_remove]
    if len(keep_cols) == 0:
        continue
    X_train = X_train_full[keep_cols]
    X_test = X_test_full[keep_cols]

    m = train_eval(X_train, X_test, y_train, y_test)
    rows.append({
        "setting": f"ablate_{group_name}",
        "removed_group": group_name,
        "n_features": len(keep_cols),
        **m
    })

results = pd.DataFrame(rows)
for metric in ["accuracy", "precision", "recall", "f1"]:
    results[f"delta_{metric}_vs_baseline"] = results[metric] - baseline_metrics[metric]

results = results.sort_values("f1", ascending=False).reset_index(drop=True)
display(results)

Total features: 26
is_broadcast
length
has_company_id
is_adv_channel
has_uuid16
len_uuid16
has_uuid128
len_data
len_ad_type
len_service_data
time_start_end
delta_end_start
delta_start_start
is_ADV_IND
is_ADV_NONCONN_IND
is_ADV_SCAN_IND
is_SCAN_REQ
is_37
is_38
is_39
time_to_next
packet_send_freq
service_data_len_ratio
ad_type_len_ratio
data_len_ratio
uuid16_len_ratio

Feature groups used for ablation:
- channel_packet_flags: 9 features
- presence_flags: 3 features
- time_to_next_and_freq: 2 features
- length_ratios_raw: 4 features


,setting,removed_group,n_features,accuracy,precision,recall,f1,delta_accuracy_vs_baseline,delta_precision_vs_baseline,delta_recall_vs_baseline,delta_f1_vs_baseline
0,baseline_all_features,,26,0.997613,0.997824,0.999075,0.998449,0.000000,0.000000,0.000000,0.000000
1,ablate_presence_flags,presence_flags,23,0.994058,0.997043,0.995228,0.996135,-0.003554,-0.000781,-0.003847,-0.002315
2,ablate_channel_packet_flags,channel_packet_flags,17,0.993040,0.993708,0.997266,0.995484,-0.004573,-0.004116,-0.001809,-0.002965
3,ablate_length_ratios_raw,length_ratios_raw,22,0.992237,0.997886,0.992010,0.994940,-0.005375,0.000063,-0.007065,-0.003510
4,ablate_time_to_next_and_freq,time_to_next_and_freq,24,0.992018,0.997601,0.992010,0.994797,-0.005595,-0.000223,-0.007065,-0.003652


In [9]:
# Single-feature ablation: leave-all-in baseline, drop one feature at a time
import pandas as pd

all_features = list(X_train_full.columns)

rows = []
baseline = train_eval(X_train_full, X_test_full, y_train, y_test)
rows.append({
    "setting": "baseline_all_features",
    "removed_feature": "",
    "n_features": len(all_features),
    **baseline
})

for feat in all_features:
    keep_cols = [c for c in all_features if c != feat]
    Xtr = X_train_full[keep_cols]
    Xte = X_test_full[keep_cols]
    m = train_eval(Xtr, Xte, y_train, y_test)
    rows.append({
        "setting": f"drop_{feat}",
        "removed_feature": feat,
        "n_features": len(keep_cols),
        **m
    })
    print("This feature is done: ", feat)

ablation_df = pd.DataFrame(rows)

for metric in ["accuracy", "precision", "recall", "f1"]:
    ablation_df[f"delta_{metric}_vs_baseline"] = ablation_df[metric] - baseline[metric]

# Most important first = biggest performance drop (most negative delta_f1)
ablation_df = ablation_df.sort_values("delta_f1_vs_baseline").reset_index(drop=True)
display(ablation_df)

ablation_df.to_csv("../code/apple_single_feature_ablation.csv", index=False)
print("Saved: ../code/apple_single_feature_ablation.csv")

This feature is done:  is_broadcast
This feature is done:  length
This feature is done:  has_company_id
This feature is done:  is_adv_channel
This feature is done:  has_uuid16
This feature is done:  len_uuid16
This feature is done:  has_uuid128
This feature is done:  len_data
This feature is done:  len_ad_type
This feature is done:  len_service_data
This feature is done:  time_start_end
This feature is done:  delta_end_start
This feature is done:  delta_start_start
This feature is done:  is_ADV_IND
This feature is done:  is_ADV_NONCONN_IND
This feature is done:  is_ADV_SCAN_IND
This feature is done:  is_SCAN_REQ
This feature is done:  is_37
This feature is done:  is_38
This feature is done:  is_39
This feature is done:  time_to_next
This feature is done:  packet_send_freq
This feature is done:  service_data_len_ratio
This feature is done:  ad_type_len_ratio
This feature is done:  data_len_ratio
This feature is done:  uuid16_len_ratio


,setting,removed_feature,n_features,accuracy,precision,recall,f1,delta_accuracy_vs_baseline,delta_precision_vs_baseline,delta_recall_vs_baseline,delta_f1_vs_baseline
0,drop_uuid16_len_ratio,uuid16_len_ratio,25,0.992007,0.997586,0.992010,0.994790,-0.005606,-0.000238,-0.007065,-0.003659
1,drop_length,length,25,0.993082,0.997930,0.993068,0.995493,-0.004530,0.000106,-0.006007,-0.002956
2,drop_is_broadcast,is_broadcast,25,0.993449,0.997993,0.993482,0.995733,-0.004163,0.000169,-0.005593,-0.002716
3,drop_is_ADV_SCAN_IND,is_ADV_SCAN_IND,25,0.994516,0.997446,0.995421,0.996432,-0.003096,-0.000378,-0.003655,-0.002017
4,drop_is_adv_channel,is_adv_channel,25,0.994534,0.997461,0.995428,0.996443,-0.003079,-0.000363,-0.003647,-0.002006
5,drop_is_SCAN_REQ,is_SCAN_REQ,25,0.994539,0.997468,0.995428,0.996447,-0.003073,-0.000356,-0.003647,-0.002002
6,drop_has_company_id,has_company_id,25,0.994562,0.997505,0.995421,0.996462,-0.003051,-0.000319,-0.003655,-0.001987
7,drop_is_ADV_IND,is_ADV_IND,25,0.994610,0.997561,0.995428,0.996493,-0.003002,-0.000263,-0.003647,-0.001956
8,drop_delta_start_start,delta_start_start,25,0.994644,0.997605,0.995428,0.996515,-0.002968,-0.000219,-0.003647,-0.001934
9,drop_delta_end_start,delta_end_start,25,0.994741,0.997731,0.995428,0.996578,-0.002871,-0.000093,-0.003647,-0.001871


Saved: ../code/apple_single_feature_ablation.csv


In [10]:
# Single-feature ablation without any is_* features
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import GroupShuffleSplit

target_list_apple = ['iPad', 'AirTag', 'AirPods']

apart_df_1 = load_data('../data/apple_labeled/apple_apartment_group1_6h_1_labeled.csv')
apart_df_2 = load_data('../data/apple_labeled/apple_apartment_group1_6h_2_labeled.csv')
apart_df_3 = load_data('../data/apple_labeled/apple_apartment_group1_6h_3_labeled.csv')
df_eval = load_data('../data/apple_labeled/apple_uni_group1_2h_labeled.csv')

df_eval['service_data'] = df_eval['service_data'].astype(apart_df_1['service_data'].dtype)

int_apart_1 = combine_data(apart_df_1, target_list_apple)
int_apart_2 = combine_data(apart_df_2, target_list_apple)
int_apart_3 = combine_data(apart_df_3, target_list_apple)

train_df = pd.concat([int_apart_1, int_apart_2, int_apart_3], ignore_index=True)

try:
    dropped_columns
except NameError:
    dropped_columns = [
        'source', 'labelled_device', 'time', 'ad_type', 'data', 'service_data',
        'uuid16', 'uuid128', 'manufacturer_data', 'rssi'
    ]

base_X = train_df.drop(columns=[c for c in dropped_columns if c in train_df.columns] + ['group_label'], errors='ignore')
base_X = base_X.select_dtypes(include=['number', 'bool']).copy()
non_is_base_X = base_X[[c for c in base_X.columns if not c.startswith('is_')]].copy()
y = train_df['group_label'].astype(int)
groups = train_df['source'] if 'source' in train_df.columns else pd.Series(np.arange(len(train_df)))

feature_groups = {
    'length': [c for c in non_is_base_X.columns if c == 'length'],
    'has_company_id': [c for c in non_is_base_X.columns if c == 'has_company_id'],
    'has_uuid16': [c for c in non_is_base_X.columns if c == 'has_uuid16'],
    'len_uuid16': [c for c in non_is_base_X.columns if c == 'len_uuid16'],
    'has_uuid128': [c for c in non_is_base_X.columns if c == 'has_uuid128'],
    'len_data': [c for c in non_is_base_X.columns if c == 'len_data'],
    'len_ad_type': [c for c in non_is_base_X.columns if c == 'len_ad_type'],
    'len_service_data': [c for c in non_is_base_X.columns if c == 'len_service_data'],
    'time_start_end': [c for c in non_is_base_X.columns if c == 'time_start_end'],
    'delta_end_start': [c for c in non_is_base_X.columns if c == 'delta_end_start'],
    'delta_start_start': [c for c in non_is_base_X.columns if c == 'delta_start_start'],
    'time_to_next': [c for c in non_is_base_X.columns if c == 'time_to_next'],
    'packet_send_freq': [c for c in non_is_base_X.columns if c == 'packet_send_freq'],
    'service_data_len_ratio': [c for c in non_is_base_X.columns if c == 'service_data_len_ratio'],
    'ad_type_len_ratio': [c for c in non_is_base_X.columns if c == 'ad_type_len_ratio'],
    'data_len_ratio': [c for c in non_is_base_X.columns if c == 'data_len_ratio'],
    'uuid16_len_ratio': [c for c in non_is_base_X.columns if c == 'uuid16_len_ratio'],
}
feature_groups = {k: v for k, v in feature_groups.items() if len(v) > 0}

non_is_features = [c for c in non_is_base_X.columns if c not in feature_groups]
print(f'Total features without is_*: {len(non_is_base_X.columns)}')
print(f'Single-feature ablation count: {len(feature_groups)}')
print(f'Kept as baseline-only extras: {len(non_is_features)}')

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(non_is_base_X, y, groups=groups))
X_train_full = non_is_base_X.iloc[train_idx]
X_test_full = non_is_base_X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

def train_eval(X_train, X_test, y_train, y_test):
    clf = RandomForestClassifier(
        random_state=1,
        class_weight='balanced',
        n_estimators=50,
        min_samples_split=10,
        min_samples_leaf=3,
        min_weight_fraction_leaf=0.002,
    )
    clf.fit(X_train, y_train)
    pred = clf.predict(X_test)
    acc = accuracy_score(y_test, pred)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average='binary', zero_division=0)
    return {'accuracy': acc, 'precision': p, 'recall': r, 'f1': f1}

rows = []
baseline_metrics = train_eval(X_train_full, X_test_full, y_train, y_test)
rows.append({'setting': 'baseline_no_is_features', 'removed_feature': '', 'n_features': X_train_full.shape[1], **baseline_metrics})

for feat, cols_to_remove in feature_groups.items():
    keep_cols = [c for c in non_is_base_X.columns if c not in cols_to_remove]
    m = train_eval(X_train_full[keep_cols], X_test_full[keep_cols], y_train, y_test)
    rows.append({'setting': f'drop_{feat}', 'removed_feature': feat, 'n_features': len(keep_cols), **m})
    print('This feature is done: ', feat)

ablation_df = pd.DataFrame(rows)
for metric in ['accuracy', 'precision', 'recall', 'f1']:
    ablation_df[f'delta_{metric}_vs_baseline'] = ablation_df[metric] - baseline_metrics[metric]
ablation_df = ablation_df.sort_values('delta_f1_vs_baseline').reset_index(drop=True)
display(ablation_df)
ablation_df.to_csv('../code/apple_single_feature_ablation_no_is.csv', index=False)
print('Saved: ../code/apple_single_feature_ablation_no_is.csv')

Total features without is_*: 17
Single-feature ablation count: 17
Kept as baseline-only extras: 0
This feature is done:  length
This feature is done:  has_company_id
This feature is done:  has_uuid16
This feature is done:  len_uuid16
This feature is done:  has_uuid128
This feature is done:  len_data
This feature is done:  len_ad_type
This feature is done:  len_service_data
This feature is done:  time_start_end
This feature is done:  delta_end_start
This feature is done:  delta_start_start
This feature is done:  time_to_next
This feature is done:  packet_send_freq
This feature is done:  service_data_len_ratio
This feature is done:  ad_type_len_ratio
This feature is done:  data_len_ratio
This feature is done:  uuid16_len_ratio


,setting,removed_feature,n_features,accuracy,precision,recall,f1,delta_accuracy_vs_baseline,delta_precision_vs_baseline,delta_recall_vs_baseline,delta_f1_vs_baseline
0,drop_time_to_next,time_to_next,16,0.981492,0.978071,0.998324,0.988094,-0.011548,-0.015637,0.001058,-0.007390
1,drop_delta_end_start,delta_end_start,16,0.988085,0.990766,0.993774,0.992268,-0.004954,-0.002943,-0.003492,-0.003216
2,drop_uuid16_len_ratio,uuid16_len_ratio,16,0.989184,0.993955,0.991973,0.992963,-0.003856,0.000247,-0.005293,-0.002521
3,drop_time_start_end,time_start_end,16,0.990470,0.993687,0.993926,0.993807,-0.002570,-0.000021,-0.003340,-0.001677
4,drop_len_ad_type,len_ad_type,16,0.990624,0.994175,0.993634,0.993904,-0.002416,0.000466,-0.003632,-0.001580
5,drop_len_data,len_data,16,0.990624,0.994175,0.993634,0.993904,-0.002416,0.000466,-0.003632,-0.001580
6,drop_length,length,16,0.990749,0.993799,0.994178,0.993988,-0.002291,0.000091,-0.003089,-0.001496
7,drop_len_service_data,len_service_data,16,0.991013,0.994207,0.994111,0.994159,-0.002026,0.000498,-0.003155,-0.001325
8,drop_len_uuid16,len_uuid16,16,0.992570,0.994032,0.996323,0.995176,-0.000470,0.000324,-0.000943,-0.000308
9,drop_has_company_id,has_company_id,16,0.992593,0.992996,0.997407,0.995196,-0.000447,-0.000713,0.000141,-0.000288


Saved: ../code/apple_single_feature_ablation_no_is.csv
